In [1]:
import os, sys

# Works whether you run from project root or notebooks/ folder
PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) == 'notebooks'
    else os.getcwd()
)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('Project root:', PROJECT_ROOT)

Project root: c:\Users\user\Desktop\New folder (3)\race_rc_project


In [2]:
import json, re
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

from preprocessing import BASE, load_race_data
print('Imports OK')

Imports OK


In [3]:
print('[1/6] Loading data...')
train_df = load_race_data('train')
val_df   = load_race_data('val')
test_df  = load_race_data('test')
print(f'  Train : {train_df.shape}')
print(f'  Val   : {val_df.shape}')
print(f'  Test  : {test_df.shape}')
print(f'  Columns: {list(train_df.columns)}')
train_df.head(2)

[1/6] Loading data...
  Train : (87852, 9)
  Val   : (87852, 9)
  Test  : (87852, 9)
  Columns: ['Unnamed: 0', 'id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer']


,Unnamed: 0,id,article,question,A,B,C,D,answer
0,0,middle7348.txt,In the summer between my first year and second...,Before the writer came to the high school summ...,instructor,camper,student,reporter,C
1,1,middle7348.txt,In the summer between my first year and second...,How many times did the writer invite the boy t...,Once,Twice,Three times,Many times,B


In [4]:
print('[Missing values]')
print(train_df.isnull().sum())
print('\n[Data types]')
print(train_df.dtypes)

[Missing values]
Unnamed: 0    0
id            0
article       0
question      0
A             0
B             0
C             0
D             0
answer        0
dtype: int64

[Data types]
Unnamed: 0    int64
id              str
article         str
question        str
A               str
B               str
C               str
D               str
answer          str
dtype: object


In [5]:
print('[2/6] Answer distribution...')
ans_counts = train_df['answer'].value_counts().sort_index()
print(ans_counts)
print(f'  Class balance A={( train_df["answer"]=="A").mean():.3f}  B={(train_df["answer"]=="B").mean():.3f}  C={(train_df["answer"]=="C").mean():.3f}  D={(train_df["answer"]=="D").mean():.3f}')

fig = px.bar(
    x=ans_counts.index, y=ans_counts.values,
    labels={'x': 'Answer Label', 'y': 'Count'},
    title='Answer Label Distribution in Training Set',
    color=ans_counts.index,
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto=True,
)
fig.show()

[2/6] Answer distribution...
answer
A    19142
B    22722
C    23887
D    22101
Name: count, dtype: int64
  Class balance A=0.218  B=0.259  C=0.272  D=0.252


In [6]:
print('[3/6] Length statistics...')
train_df['article_len']  = train_df['article'].str.split().str.len()
train_df['question_len'] = train_df['question'].str.split().str.len()
train_df['option_a_len'] = train_df['A'].str.split().str.len()

print('Article length stats (words):')
print(train_df['article_len'].describe().round(1))
print('\nQuestion length stats (words):')
print(train_df['question_len'].describe().round(1))

fig1 = px.histogram(train_df, x='article_len', title='Article Length Distribution',
                    labels={'article_len': 'Words in Article'}, nbins=50,
                    color_discrete_sequence=['steelblue'])
fig1.show()

fig2 = px.histogram(train_df, x='question_len', title='Question Length Distribution',
                    labels={'question_len': 'Words in Question'}, nbins=30,
                    color_discrete_sequence=['coral'])
fig2.show()

[3/6] Length statistics...
Article length stats (words):
count    87852.0
mean       275.0
std         97.9
min          2.0
25%        217.0
50%        279.0
75%        326.0
max       1162.0
Name: article_len, dtype: float64

Question length stats (words):
count    87852.0
mean        10.0
std          3.4
min          1.0
25%          8.0
50%         10.0
75%         12.0
max         63.0
Name: question_len, dtype: float64


In [7]:
def detect_question_type(question):
    q = str(question).strip().lower()
    for wh in ['what', 'who', 'where', 'when', 'why', 'how', 'which']:
        if q.startswith(wh):
            return wh
    return 'other'

print('[4/6] Question type distribution...')
train_df['question_type'] = train_df['question'].apply(detect_question_type)
qt_counts = train_df['question_type'].value_counts()
print(qt_counts)

fig = px.pie(values=qt_counts.values, names=qt_counts.index,
             title='Question Type Distribution',
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.show()

[4/6] Question type distribution...
question_type
other    47983
what     16894
which    10828
why       4114
how       3447
when      2249
who       1234
where     1103
Name: count, dtype: int64


In [8]:
def get_top_words(series, top_n=20):
    all_words = []
    for text in series.dropna():
        text_clean = re.sub(r'[^a-z\s]', ' ', str(text).lower())
        words = [w for w in text_clean.split() if w not in STOPWORDS and len(w) > 2]
        all_words.extend(words)
    return Counter(all_words).most_common(top_n)

print('[5/6] Top words in articles...')
top_words = get_top_words(train_df['article'])
words_df  = pd.DataFrame(top_words, columns=['Word', 'Count'])

fig = px.bar(words_df, x='Count', y='Word', orientation='h',
             title='Top 20 Words in Articles (stopwords removed)',
             color='Count', color_continuous_scale='Blues')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

[5/6] Top words in articles...


In [9]:
# Outlier detection
q95_art  = train_df['article_len'].quantile(0.95)
q05_art  = train_df['article_len'].quantile(0.05)
outliers = train_df[(train_df['article_len'] > q95_art) | (train_df['article_len'] < q05_art)]
print(f'Outliers (article length outside 5th-95th percentile): {len(outliers)}')

# Feature correlation
corr = train_df[['article_len','question_len','option_a_len']].corr()
print('\nCorrelation matrix:')
print(corr.round(3))

Outliers (article length outside 5th-95th percentile): 8712

Correlation matrix:
              article_len  question_len  option_a_len
article_len         1.000         0.182         0.251
question_len        0.182         1.000        -0.029
option_a_len        0.251        -0.029         1.000


In [10]:
print('[6/6] Saving EDA summary...')
summary = {
    'total_train': len(train_df),
    'total_val':   len(val_df),
    'total_test':  len(test_df),
    'avg_article_len_words':  round(train_df['article_len'].mean(), 1),
    'avg_question_len_words': round(train_df['question_len'].mean(), 1),
    'answer_balance_A': round((train_df['answer'] == 'A').mean(), 3),
    'answer_balance_B': round((train_df['answer'] == 'B').mean(), 3),
    'answer_balance_C': round((train_df['answer'] == 'C').mean(), 3),
    'answer_balance_D': round((train_df['answer'] == 'D').mean(), 3),
    'outlier_count': len(outliers),
}

processed_dir = os.path.join(BASE, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
summary_path = os.path.join(processed_dir, 'eda_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'EDA complete. Summary saved to {summary_path}')
print(json.dumps(summary, indent=2))

[6/6] Saving EDA summary...
EDA complete. Summary saved to c:\Users\user\Desktop\New folder (3)\race_rc_project\data\processed\eda_summary.json
{
  "total_train": 87852,
  "total_val": 87852,
  "total_test": 87852,
  "avg_article_len_words": 275.0,
  "avg_question_len_words": 10.0,
  "answer_balance_A": 0.218,
  "answer_balance_B": 0.259,
  "answer_balance_C": 0.272,
  "answer_balance_D": 0.252,
  "outlier_count": 8712
}
